#Cargue de información

In [ ]:
# 1. CARGA DE DATOS
import pandas as pd
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

df_fc = pd.read_parquet(ruta_fasecolda)
df_ca = pd.read_parquet(ruta_canales)

In [ ]:
df_fc.info()
df_ca.info()

# Tablero anual por Ramo/ año

In [ ]:
# ==============================================================================
# BLOQUE 8.2: TABLERO ANUAL - RAMO (INDUSTRIA, CANALES, BOLIVAR)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS Y LÍMITES TEMPORALES
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_ca = pd.read_parquet(ruta_canales)

    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
    df_ca['FECHA'] = pd.to_datetime(df_ca['FECHA'], errors='coerce')

    fecha_maxima_comun = min(df_fc['FECHA'].max(), df_ca['FECHA'].max())
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted([a for a in df_fc['AÑO'].dropna().unique() if a <= fecha_maxima_comun.year])
ramos_unicos = sorted(list(set(df_fc['RAMOS'].dropna().unique()) | set(df_ca['RAMOS'].dropna().unique())))

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_txt = meses_dict.get(fecha_maxima_comun.month, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_ramo = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
sel_anios_atras_ramo = widgets.SelectMultiple(options=[1, 2, 3, 4, 5], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
dd_mes_ramo = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_ramos_ramo = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '150px'})

btn_ramo = widgets.Button(description='Generar Tablero Ramo', button_style='primary', icon='list', layout={'width': '300px', 'height': '40px'})

ui_ramo = widgets.VBox([
    widgets.HBox([dd_anio_ramo, dd_mes_ramo, sel_anios_atras_ramo]),
    sel_ramos_ramo,
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Límite de datos compartidos detectado: <b>{fecha_maxima_comun.strftime('%B %Y')}</b>. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_ramo
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_ramo = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_ramo(b):
    with out_ramo:
        clear_output(wait=True)

        anio_base = dd_anio_ramo.value
        anios_atras_lista = list(sel_anios_atras_ramo.value)
        ramo_obj = list(sel_ramos_ramo.value)
        mes_str = dd_mes_ramo.value
        mes_num = meses_inversos[mes_str]

        if not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Ramo o Años Atrás)."))
            return

        display(Markdown(f"# 📊 % VAR POR AÑO Y RAMO (Acumulado a {mes_str})"))
        display(Markdown("---"))

        def fmt_m(v): return "-" if pd.isna(v) else f"${v:,.0f}"
        def fmt_p(v): return "-" if pd.isna(v) else f"{v:+.2%}"
        def fmt_v(v): return "-" if pd.isna(v) else f"{v:.2f}x"

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            m_fc = (df_fc['FECHA'].dt.month == mes_num) & (df_fc['RAMOS'].isin(ramo_obj)) & (df_fc['AÑO'].isin([anio_base, anio_ant]))
            m_ca = (df_ca['FECHA'].dt.month <= mes_num) & (df_ca['RAMOS'].isin(ramo_obj)) & (df_ca['AÑO'].isin([anio_base, anio_ant]))

            df_fc_fil = df_fc[m_fc]
            df_ca_fil = df_ca[m_ca]

            if df_fc_fil.empty:
                display(Markdown(f"⚠️ No hay datos en Industria para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'RAMOS'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            bol = df_fc_fil[df_fc_fil['COMPANIA'] == 'BOLIVAR'].groupby(['AÑO', 'RAMOS'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            can = df_ca_fil.groupby(['AÑO', 'RAMOS'])['REAL'].sum().unstack('AÑO').fillna(0)

            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = 0
                if a not in bol.columns: bol[a] = 0
                if a not in can.columns: can[a] = 0

            cols = pd.MultiIndex.from_tuples([
                ('INDUSTRIA', lbl_anterior), ('INDUSTRIA', lbl_actual), ('INDUSTRIA', 'Var'),
                ('CANAL', lbl_anterior), ('CANAL', lbl_actual), ('CANAL', 'Var'), ('CANAL', 'Veces'),
                ('BOLIVAR', lbl_anterior), ('BOLIVAR', lbl_actual), ('BOLIVAR', 'Var'), ('BOLIVAR', 'Veces')
            ])
            res = pd.DataFrame(index=mkt.index, columns=cols)

            res[('INDUSTRIA', lbl_anterior)] = mkt.get(anio_ant, pd.Series(0, index=mkt.index))
            res[('INDUSTRIA', lbl_actual)] = mkt.get(anio_base, pd.Series(0, index=mkt.index))
            res[('INDUSTRIA', 'Var')] = (res[('INDUSTRIA', lbl_actual)] / res[('INDUSTRIA', lbl_anterior)].replace(0, np.nan)) - 1

            res[('CANAL', lbl_anterior)] = can.get(anio_ant, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('CANAL', lbl_actual)] = can.get(anio_base, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('CANAL', 'Var')] = (res[('CANAL', lbl_actual)] / res[('CANAL', lbl_anterior)].replace(0, np.nan)) - 1
            res[('CANAL', 'Veces')] = res[('CANAL', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            res[('BOLIVAR', lbl_anterior)] = bol.get(anio_ant, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('BOLIVAR', lbl_actual)] = bol.get(anio_base, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('BOLIVAR', 'Var')] = (res[('BOLIVAR', lbl_actual)] / res[('BOLIVAR', lbl_anterior)].replace(0, np.nan)) - 1
            res[('BOLIVAR', 'Veces')] = res[('BOLIVAR', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            orden_filas = sorted([r for r in ramo_obj if r in res.index])
            res = res.loc[orden_filas]

            tot_series = res.sum()
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                ant_tot = tot_series.get((ent, lbl_anterior), 0)
                act_tot = tot_series.get((ent, lbl_actual), 0)
                tot_series[(ent, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

            v_m_tot = tot_series.get(('INDUSTRIA', 'Var'), np.nan)
            tot_series[('CANAL', 'Veces')] = (tot_series.get(('CANAL', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            tot_series[('BOLIVAR', 'Veces')] = (tot_series.get(('BOLIVAR', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan

            res.loc['TOTAL'] = tot_series

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))
            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DE ESTILOS Y MEMORIA
            # ==================================================================
            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                    if (ent, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(ent, 'Var')], errors='coerce')
                        df_st[(ent, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (ent, 'Veces') in data.columns:
                        v_vec = pd.to_numeric(data[(ent, 'Veces')], errors='coerce')
                        df_st[(ent, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

                    m_sin = data[(ent, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (ent, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (ent, 'Veces') in data.columns: df_st.loc[m_sin, (ent, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'td:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_ramo.on_click(generar_tablero_ramo)
display(ui_ramo, out_ramo)

#Por Sucursal

In [ ]:
# ==============================================================================
# BLOQUE 8.5: TABLERO ANUAL - SUCURSAL (INDUSTRIA, CANALES, BOLIVAR)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS Y LÍMITES TEMPORALES
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_ca = pd.read_parquet(ruta_canales)

    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
    df_ca['FECHA'] = pd.to_datetime(df_ca['FECHA'], errors='coerce')

    fecha_maxima_comun = min(df_fc['FECHA'].max(), df_ca['FECHA'].max())
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted([a for a in df_fc['AÑO'].dropna().unique() if a <= fecha_maxima_comun.year])
ramos_unicos = sorted(list(set(df_fc['RAMOS'].dropna().unique()) | set(df_ca['RAMOS'].dropna().unique())))
sucursales_unicas = sorted(list(set(df_fc['SUCURSAL'].dropna().unique()) | set(df_ca['SUCURSAL'].dropna().unique())))

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_txt = meses_dict.get(fecha_maxima_comun.month, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_suc = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_suc = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_suc = widgets.SelectMultiple(options=[1, 2, 3, 4, 5], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_ramos_suc = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})
sel_sucursales = widgets.SelectMultiple(options=sucursales_unicas, value=[], description='📍 Sucursales:', layout={'width': '350px', 'height': '120px'})

btn_suc = widgets.Button(description='Generar Tablero Sucursal', button_style='primary', icon='sitemap', layout={'width': '300px', 'height': '40px'})

ui_suc = widgets.VBox([
    widgets.HBox([dd_anio_suc, dd_mes_suc, sel_anios_atras_suc]),
    widgets.HBox([sel_sucursales, sel_ramos_suc]),
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Límite de datos compartidos detectado: <b>{fecha_maxima_comun.strftime('%B %Y')}</b>. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_suc
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_suc = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_sucursal(b):
    with out_suc:
        clear_output(wait=True)

        anio_base = dd_anio_suc.value
        anios_atras_lista = list(sel_anios_atras_suc.value)
        ramo_obj = list(sel_ramos_suc.value)
        suc_obj = list(sel_sucursales.value)
        mes_str = dd_mes_suc.value
        mes_num = meses_inversos[mes_str]

        if not ramo_obj or not suc_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Ramo, Sucursal o Años Atrás)."))
            return

        display(Markdown(f"# 📊 % VAR POR AÑO Y SUCURSAL (Acumulado a {mes_str})"))
        display(Markdown("---"))

        def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
        def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
        def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            m_fc = (df_fc['FECHA'].dt.month == mes_num) & (df_fc['RAMOS'].isin(ramo_obj)) & (df_fc['SUCURSAL'].isin(suc_obj)) & (df_fc['AÑO'].isin([anio_base, anio_ant]))
            m_ca = (df_ca['FECHA'].dt.month <= mes_num) & (df_ca['RAMOS'].isin(ramo_obj)) & (df_ca['SUCURSAL'].isin(suc_obj)) & (df_ca['AÑO'].isin([anio_base, anio_ant]))

            df_fc_fil = df_fc[m_fc]
            df_ca_fil = df_ca[m_ca]

            if df_fc_fil.empty and df_ca_fil.empty:
                display(Markdown(f"⚠️ No hay datos en ninguna base para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'SUCURSAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            bol = df_fc_fil[df_fc_fil['COMPANIA'] == 'BOLIVAR'].groupby(['AÑO', 'SUCURSAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            can = df_ca_fil.groupby(['AÑO', 'SUCURSAL'])['REAL'].sum().unstack('AÑO').fillna(0)

            todos_indices = sorted(list(set(mkt.index) | set(bol.index) | set(can.index)))

            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = pd.Series(0, index=mkt.index)
                if a not in bol.columns: bol[a] = pd.Series(0, index=bol.index)
                if a not in can.columns: can[a] = pd.Series(0, index=can.index)

            cols = pd.MultiIndex.from_tuples([
                ('INDUSTRIA', lbl_anterior), ('INDUSTRIA', lbl_actual), ('INDUSTRIA', 'Var'),
                ('CANAL', lbl_anterior), ('CANAL', lbl_actual), ('CANAL', 'Var'), ('CANAL', 'Veces'),
                ('BOLIVAR', lbl_anterior), ('BOLIVAR', lbl_actual), ('BOLIVAR', 'Var'), ('BOLIVAR', 'Veces')
            ])
            res = pd.DataFrame(index=todos_indices, columns=cols)

            # Llenado
            res[('INDUSTRIA', lbl_anterior)] = mkt[anio_ant].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', lbl_actual)] = mkt[anio_base].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', 'Var')] = (res[('INDUSTRIA', lbl_actual)] / res[('INDUSTRIA', lbl_anterior)].replace(0, np.nan)) - 1

            res[('CANAL', lbl_anterior)] = can[anio_ant].reindex(todos_indices).fillna(0)
            res[('CANAL', lbl_actual)] = can[anio_base].reindex(todos_indices).fillna(0)
            res[('CANAL', 'Var')] = (res[('CANAL', lbl_actual)] / res[('CANAL', lbl_anterior)].replace(0, np.nan)) - 1
            res[('CANAL', 'Veces')] = res[('CANAL', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            res[('BOLIVAR', lbl_anterior)] = bol[anio_ant].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', lbl_actual)] = bol[anio_base].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', 'Var')] = (res[('BOLIVAR', lbl_actual)] / res[('BOLIVAR', lbl_anterior)].replace(0, np.nan)) - 1
            res[('BOLIVAR', 'Veces')] = res[('BOLIVAR', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            orden_filas = sorted([r for r in suc_obj if r in res.index])
            res = res.loc[orden_filas]

            # Cálculo de la fila TOTAL
            tot_series = res.sum()
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                ant_tot = tot_series.get((ent, lbl_anterior), 0)
                act_tot = tot_series.get((ent, lbl_actual), 0)
                tot_series[(ent, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

            v_m_tot = tot_series.get(('INDUSTRIA', 'Var'), np.nan)
            tot_series[('CANAL', 'Veces')] = (tot_series.get(('CANAL', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            tot_series[('BOLIVAR', 'Veces')] = (tot_series.get(('BOLIVAR', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            res.loc['TOTAL'] = tot_series

            # --- APLICAR TEXTO "Sin registros" ---
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                mask = (res[(ent, lbl_anterior)] == 0) & (res[(ent, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(ent, lbl_anterior), (ent, lbl_actual), (ent, 'Var')]
                    if (ent, 'Veces') in res.columns: cols_to_cast.append((ent, 'Veces'))
                    for col in cols_to_cast:
                        res[col] = res[col].astype(object)
                    res.loc[mask, [(ent, lbl_anterior), (ent, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (ent, 'Var')] = "-"
                    if (ent, 'Veces') in res.columns: res.loc[mask, (ent, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))
            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            def estilo_celdas(data):
                # Usamos dtype=object
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                    if (ent, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(ent, 'Var')], errors='coerce')
                        # Listas de comprensión
                        df_st[(ent, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (ent, 'Veces') in data.columns:
                        v_vec = pd.to_numeric(data[(ent, 'Veces')], errors='coerce')
                        # Listas de comprensión
                        df_st[(ent, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

                    m_sin = data[(ent, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (ent, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (ent, 'Veces') in data.columns: df_st.loc[m_sin, (ent, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    # Forzar el texto en el index TOTAL
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'td:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_suc.on_click(generar_tablero_sucursal)
display(ui_suc, out_suc)

#Por Regional

In [ ]:
# ==============================================================================
# BLOQUE 8.3: TABLERO ANUAL - REGIONAL (INDUSTRIA, CANALES, BOLIVAR)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS Y LÍMITES TEMPORALES
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_ca = pd.read_parquet(ruta_canales)

    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
    df_ca['FECHA'] = pd.to_datetime(df_ca['FECHA'], errors='coerce')

    fecha_maxima_comun = min(df_fc['FECHA'].max(), df_ca['FECHA'].max())
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted([a for a in df_fc['AÑO'].dropna().unique() if a <= fecha_maxima_comun.year])
ramos_unicos = sorted(list(set(df_fc['RAMOS'].dropna().unique()) | set(df_ca['RAMOS'].dropna().unique())))
regionales_unicas = sorted(list(set(df_fc['REGIONAL'].dropna().unique()) | set(df_ca['REGIONAL'].dropna().unique())))

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_txt = meses_dict.get(fecha_maxima_comun.month, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_reg = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_reg = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_reg = widgets.SelectMultiple(options=[1, 2, 3, 4, 5], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_ramos_reg = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})
sel_reg_reg = widgets.SelectMultiple(options=regionales_unicas, value=[], description='🗺️ Regionales:', layout={'width': '350px', 'height': '120px'})

btn_reg = widgets.Button(description='Generar Tablero Regional', button_style='primary', icon='map', layout={'width': '300px', 'height': '40px'})

ui_reg = widgets.VBox([
    widgets.HBox([dd_anio_reg, dd_mes_reg, sel_anios_atras_reg]),
    widgets.HBox([sel_reg_reg, sel_ramos_reg]),
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Límite de datos compartidos detectado: <b>{fecha_maxima_comun.strftime('%B %Y')}</b>. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_reg
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_reg = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_regional(b):
    with out_reg:
        clear_output(wait=True)

        anio_base = dd_anio_reg.value
        anios_atras_lista = list(sel_anios_atras_reg.value)
        ramo_obj = list(sel_ramos_reg.value)
        reg_obj = list(sel_reg_reg.value)
        mes_str = dd_mes_reg.value
        mes_num = meses_inversos[mes_str]

        if not ramo_obj or not reg_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Ramo, Regional o Años Atrás)."))
            return

        display(Markdown(f"# 📊 % VAR POR AÑO Y REGIONAL (Acumulado a {mes_str})"))
        display(Markdown("---"))

        def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
        def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
        def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            m_fc = (df_fc['FECHA'].dt.month == mes_num) & (df_fc['RAMOS'].isin(ramo_obj)) & (df_fc['REGIONAL'].isin(reg_obj)) & (df_fc['AÑO'].isin([anio_base, anio_ant]))
            m_ca = (df_ca['FECHA'].dt.month <= mes_num) & (df_ca['RAMOS'].isin(ramo_obj)) & (df_ca['REGIONAL'].isin(reg_obj)) & (df_ca['AÑO'].isin([anio_base, anio_ant]))

            df_fc_fil = df_fc[m_fc]
            df_ca_fil = df_ca[m_ca]

            if df_fc_fil.empty and df_ca_fil.empty:
                display(Markdown(f"⚠️ No hay datos en ninguna base para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'REGIONAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            bol = df_fc_fil[df_fc_fil['COMPANIA'] == 'BOLIVAR'].groupby(['AÑO', 'REGIONAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            can = df_ca_fil.groupby(['AÑO', 'REGIONAL'])['REAL'].sum().unstack('AÑO').fillna(0)

            # Unión inteligente de índices para que "Gerencia" o similares no se pierdan
            todos_indices = sorted(list(set(mkt.index) | set(bol.index) | set(can.index)))

            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = pd.Series(0, index=mkt.index)
                if a not in bol.columns: bol[a] = pd.Series(0, index=bol.index)
                if a not in can.columns: can[a] = pd.Series(0, index=can.index)

            cols = pd.MultiIndex.from_tuples([
                ('INDUSTRIA', lbl_anterior), ('INDUSTRIA', lbl_actual), ('INDUSTRIA', 'Var'),
                ('CANAL', lbl_anterior), ('CANAL', lbl_actual), ('CANAL', 'Var'), ('CANAL', 'Veces'),
                ('BOLIVAR', lbl_anterior), ('BOLIVAR', lbl_actual), ('BOLIVAR', 'Var'), ('BOLIVAR', 'Veces')
            ])
            res = pd.DataFrame(index=todos_indices, columns=cols)

            # Llenado
            res[('INDUSTRIA', lbl_anterior)] = mkt[anio_ant].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', lbl_actual)] = mkt[anio_base].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', 'Var')] = (res[('INDUSTRIA', lbl_actual)] / res[('INDUSTRIA', lbl_anterior)].replace(0, np.nan)) - 1

            res[('CANAL', lbl_anterior)] = can[anio_ant].reindex(todos_indices).fillna(0)
            res[('CANAL', lbl_actual)] = can[anio_base].reindex(todos_indices).fillna(0)
            res[('CANAL', 'Var')] = (res[('CANAL', lbl_actual)] / res[('CANAL', lbl_anterior)].replace(0, np.nan)) - 1
            res[('CANAL', 'Veces')] = res[('CANAL', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            res[('BOLIVAR', lbl_anterior)] = bol[anio_ant].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', lbl_actual)] = bol[anio_base].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', 'Var')] = (res[('BOLIVAR', lbl_actual)] / res[('BOLIVAR', lbl_anterior)].replace(0, np.nan)) - 1
            res[('BOLIVAR', 'Veces')] = res[('BOLIVAR', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            orden_filas = sorted([r for r in reg_obj if r in res.index])
            res = res.loc[orden_filas]

            # Cálculo de la fila TOTAL
            tot_series = res.sum()
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                ant_tot = tot_series.get((ent, lbl_anterior), 0)
                act_tot = tot_series.get((ent, lbl_actual), 0)
                tot_series[(ent, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

            v_m_tot = tot_series.get(('INDUSTRIA', 'Var'), np.nan)
            tot_series[('CANAL', 'Veces')] = (tot_series.get(('CANAL', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            tot_series[('BOLIVAR', 'Veces')] = (tot_series.get(('BOLIVAR', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            res.loc['TOTAL'] = tot_series

            # --- APLICAR TEXTO "Sin registros" ---
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                mask = (res[(ent, lbl_anterior)] == 0) & (res[(ent, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(ent, lbl_anterior), (ent, lbl_actual), (ent, 'Var')]
                    if (ent, 'Veces') in res.columns: cols_to_cast.append((ent, 'Veces'))
                    for col in cols_to_cast:
                        res[col] = res[col].astype(object)
                    res.loc[mask, [(ent, lbl_anterior), (ent, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (ent, 'Var')] = "-"
                    if (ent, 'Veces') in res.columns: res.loc[mask, (ent, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))
            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DE ESTILOS Y MEMORIA
            # ==================================================================
            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                    if (ent, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(ent, 'Var')], errors='coerce')
                        df_st[(ent, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (ent, 'Veces') in data.columns:
                        v_vec = pd.to_numeric(data[(ent, 'Veces')], errors='coerce')
                        df_st[(ent, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

                    m_sin = data[(ent, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (ent, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (ent, 'Veces') in data.columns: df_st.loc[m_sin, (ent, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'td:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_reg.on_click(generar_tablero_regional)
display(ui_reg, out_reg)